In [7]:
import pandas as pd
import numpy as np
import sys
import os

# Импорт нашего кастомного трансформера из папки src
sys.path.append(os.path.abspath('.'))
from src.custom_transformers import MissingValueAdder

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [8]:
DATA_PATH = '../data/raw/titanic.csv'
df = pd.read_csv(DATA_PATH)

# Удаляем колонки, которые не нужны для модели (слишком много уникальных значений)
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df = df.drop(columns=drop_cols)

print(f"Размер данных: {df.shape}")
df.head()

Размер данных: (891, 8)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [9]:
TARGET_COL = 'Survived'

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# stratify=y сохранит пропорцию выживших в train и test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Определяем числовые и категориальные колонки
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Добавляем нашу новую фичу в список числовых
num_cols_with_missing = num_cols + ['missing_count']

print(f"Числовые: {num_cols}")
print(f"Категориальные: {cat_cols}")

Числовые: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Категориальные: ['Sex', 'Embarked']


C:\Users\yekat\AppData\Local\Temp\ipykernel_30328\3782470900.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


In [10]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Заполняем пропуски в Age медианой
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Заполняем пропуски в Embarked модой
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols_with_missing),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# Полный Pipeline с кастомным трансформером
full_pipeline = Pipeline(steps=[
    ('add_missing_feature', MissingValueAdder()), 
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

In [11]:
param_grid = {
    'classifier__C': [0.1, 1.0, 10.0],
    'classifier__solver': ['lbfgs', 'liblinear'],
    'preprocessor__num__imputer__strategy': ['mean', 'median']
}

grid_search = GridSearchCV(
    full_pipeline, 
    param_grid, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1,
    verbose=1
)

print("Обучение...")
grid_search.fit(X_train, y_train)
print("Готово!")

Обучение...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Готово!


In [12]:
print("Лучшие параметры:", grid_search.best_params_)
print("Лучшая точность (CV):", grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred))
print(f"Accuracy на тесте: {accuracy_score(y_test, y_pred):.4f}")

Лучшие параметры: {'classifier__C': 10.0, 'classifier__solver': 'lbfgs', 'preprocessor__num__imputer__strategy': 'mean'}
Лучшая точность (CV): 0.8047843610017846

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179

Accuracy на тесте: 0.8156
